In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

base_path = "/content/drive/MyDrive"

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

In [ ]:
import pandas as pd

files = [
    "batteries.csv",
    "city_daily_context.csv",
    "fleet_partners.csv",
    "riders.csv",
    "station_hourly_status.csv",
    "stations.csv",
    "support_tickets.csv",
    "swap_events.csv"
]

for file in files:
    print(file)

batteries.csv
city_daily_context.csv
fleet_partners.csv
riders.csv
station_hourly_status.csv
stations.csv
support_tickets.csv
swap_events.csv


In [ ]:
import os

base_path = "/content/drive/MyDrive"

files = [
    "batteries.csv",
    "city_daily_context.csv",
    "fleet_partners.csv",
    "riders.csv",
    "station_hourly_status.csv",
    "stations.csv",
    "support_tickets.csv",
    "swap_events.csv"
]

for file in files:
    path = os.path.join(base_path, file)
    print(file, "→", os.path.exists(path))

batteries.csv → False
city_daily_context.csv → False
fleet_partners.csv → False
riders.csv → False
station_hourly_status.csv → False
stations.csv → False
support_tickets.csv → False
swap_events.csv → False


In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    csv_files = [f for f in files if f.endswith(".csv")]
    if csv_files:
        print("FOLDER:", root)
        for f in csv_files:
            print("  ", f)

FOLDER: /content/drive/MyDrive
   swap_events.csv
   support_tickets.csv
   stations.csv
   station_hourly_status.csv
   riders.csv
   fleet_partners.csv
   city_daily_context.csv
   batteries.csv


In [ ]:
import pandas as pd

base_path = "/content/drive/MyDrive"

batteries = pd.read_csv(base_path + "/batteries.csv")
city = pd.read_csv(base_path + "/city_daily_context.csv")
fleet = pd.read_csv(base_path + "/fleet_partners.csv")
riders = pd.read_csv(base_path + "/riders.csv")
station_status = pd.read_csv(base_path + "/station_hourly_status.csv")
stations = pd.read_csv(base_path + "/stations.csv")
tickets = pd.read_csv(base_path + "/support_tickets.csv")
swaps = pd.read_csv(base_path + "/swap_events.csv")

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [ ]:
datasets = {
    "swaps": swaps,
    "stations": stations,
    "station_status": station_status,
    "riders": riders,
    "batteries": batteries,
    "tickets": tickets,
    "city": city,
    "fleet": fleet
}

for name, df in datasets.items():
    print(f"{name:15} → Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

swaps           → Rows: 3,877,013 | Columns: 22
stations        → Rows: 152 | Columns: 22
station_status  → Rows: 1,487,712 | Columns: 14
riders          → Rows: 20,000 | Columns: 12
batteries       → Rows: 6,500 | Columns: 13
tickets         → Rows: 44,000 | Columns: 12
city            → Rows: 3,282 | Columns: 12
fleet           → Rows: 12 | Columns: 12


In [ ]:
swaps.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3877013 entries, 0 to 3877012
Data columns (total 22 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   event_id                object 
 1   rider_id                object 
 2   station_id              object 
 3   event_ts                object 
 4   event_type              object 
 5   attempt_seq             int64  
 6   queue_wait_sec          int64  
 7   battery_in_id           object 
 8   battery_out_id          object 
 9   soc_in_pct              float64
 10  soh_in_pct              float64
 11  soc_out_pct             float64
 12  soh_out_pct             float64
 13  km_since_last_swap      float64
 14  tariff_code             object 
 15  list_price_inr          float64
 16  discount_inr            float64
 17  amount_charged_inr      float64
 18  payment_mode            object 
 19  energy_to_recharge_kwh  float64
 20  station_firmware        object 
 21  sync_mode               object 

In [ ]:
# Missing values
missing = swaps.isnull().sum()
missing_pct = (missing / len(swaps) * 100).round(2)

quality = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": missing_pct
})

quality[quality["Missing_Count"] > 0].sort_values(
    "Missing_Count", ascending=False
)

,Missing_Count,Missing_Percentage
battery_out_id,231204,5.96
energy_to_recharge_kwh,231204,5.96
soh_out_pct,231204,5.96
soc_out_pct,231204,5.96
soh_in_pct,28282,0.73
soc_in_pct,28282,0.73
battery_in_id,28282,0.73
km_since_last_swap,28282,0.73


In [ ]:
print("Total rows:", len(swaps))
print("Duplicate event_id:", swaps["event_id"].duplicated().sum())
print("Duplicate full rows:", swaps.duplicated().sum())

Total rows: 3877013
Duplicate event_id: 0
Duplicate full rows: 0


In [ ]:
print(swaps["event_type"].value_counts())

event_type
swap_completed               3645809
failed_no_charged_battery     134389
abandoned_queue                68533
cancelled_by_rider             16712
failed_system_error            11570
Name: count, dtype: int64


In [ ]:
total = len(swaps)
completed = (swaps["event_type"] == "swap_completed").sum()

failure_rate = (1 - completed / total) * 100

print(f"Total Attempts : {total:,}")
print(f"Completed      : {completed:,}")
print(f"Failed/Other   : {total-completed:,}")
print(f"Failure Rate   : {failure_rate:.2f}%")

Total Attempts : 3,877,013
Completed      : 3,645,809
Failed/Other   : 231,204
Failure Rate   : 5.96%


In [ ]:
swaps["event_ts"] = pd.to_datetime(swaps["event_ts"])

swaps["month"] = swaps["event_ts"].dt.to_period("M").astype(str)

monthly = swaps.groupby("month").agg(
    total_attempts=("event_id", "count"),
    completed_swaps=("event_type", lambda x: (x == "swap_completed").sum()),
    revenue=("amount_charged_inr", "sum")
).reset_index()

monthly["failure_rate"] = (
    (monthly["total_attempts"] - monthly["completed_swaps"])
    / monthly["total_attempts"] * 100
)

monthly

,month,total_attempts,completed_swaps,revenue,failure_rate
0,2024-01,110348,105490,6318995.20,4.402436
1,2024-02,111531,106582,6387614.80,4.437331
2,2024-03,123900,118444,7093617.00,4.403551
3,2024-04,131818,122836,7370255.80,6.813940
4,2024-05,146278,128328,7689594.00,12.271155
5,2024-06,148888,133743,8003275.60,10.172076
6,2024-07,158688,151824,9879524.60,4.325469
7,2024-08,172295,164931,10724296.65,4.274065
8,2024-09,195885,187302,12115209.50,4.381653
9,2024-10,229901,219838,14577271.65,4.377101


In [ ]:
failure_by_type = swaps["event_type"].value_counts().reset_index()
failure_by_type.columns = ["event_type", "count"]

failure_by_type

,event_type,count
0,swap_completed,3645809
1,failed_no_charged_battery,134389
2,abandoned_queue,68533
3,cancelled_by_rider,16712
4,failed_system_error,11570


In [ ]:
failures = swaps[swaps["event_type"] != "swap_completed"]

failure_summary = (
    failures["event_type"]
    .value_counts()
    .rename_axis("failure_type")
    .reset_index(name="count")
)

failure_summary["percentage"] = (
    failure_summary["count"] / len(failures) * 100
).round(2)

failure_summary

,failure_type,count,percentage
0,failed_no_charged_battery,134389,58.13
1,abandoned_queue,68533,29.64
2,cancelled_by_rider,16712,7.23
3,failed_system_error,11570,5.00


In [ ]:
city_performance = (
    swaps.groupby(swaps["station_id"])
    .agg(
        attempts=("event_id", "count"),
        completed=("event_type", lambda x: (x == "swap_completed").sum())
    )
    .reset_index()
)

city_performance = city_performance.merge(
    stations[["station_id", "city"]],
    on="station_id",
    how="left"
)

city_performance = (
    city_performance.groupby("city")
    .agg(
        attempts=("attempts", "sum"),
        completed=("completed", "sum")
    )
    .reset_index()
)

city_performance["failure_rate"] = (
    (city_performance["attempts"] - city_performance["completed"])
    / city_performance["attempts"] * 100
).round(2)

city_performance.sort_values("failure_rate", ascending=False)

,city,attempts,completed,failure_rate
3,Jaipur,487634,449366,7.85
1,Delhi NCR,768913,712385,7.35
2,Hyderabad,711930,664312,6.69
5,Pune,552910,525331,4.99
0,Bengaluru,825327,787697,4.56
4,Mumbai,530299,506718,4.45


In [ ]:
station_performance = (
    swaps.groupby("station_id")
    .agg(
        attempts=("event_id", "count"),
        completed=("event_type", lambda x: (x == "swap_completed").sum())
    )
    .reset_index()
)

station_performance["failure_rate"] = (
    (station_performance["attempts"] - station_performance["completed"])
    / station_performance["attempts"] * 100
).round(2)

station_performance = station_performance.merge(
    stations[[
        "station_id",
        "city",
        "location_type",
        "charger_generation",
        "slots_2w",
        "slots_3w"
    ]],
    on="station_id",
    how="left"
)

station_performance.sort_values(
    "failure_rate", ascending=False
).head(15)

,station_id,attempts,completed,failure_rate,city,location_type,charger_generation,slots_2w,slots_3w
36,STN-DEL-037,30838,28039,9.08,Delhi NCR,commercial_hub,Gen1,16,0
47,STN-DEL-048,34646,31535,8.98,Delhi NCR,market,Gen1,16,0
92,STN-JAI-137,44371,40490,8.75,Jaipur,commercial_hub,Gen1,8,0
97,STN-JAI-142,33319,30405,8.75,Jaipur,commercial_hub,Gen1,12,6
96,STN-JAI-141,45594,41613,8.73,Jaipur,commercial_hub,Gen1,16,0
44,STN-DEL-045,32237,29426,8.72,Delhi NCR,commercial_hub,Gen1,16,0
91,STN-JAI-136,29340,26785,8.71,Jaipur,transit_hub,Gen1,8,6
93,STN-JAI-138,48945,44703,8.67,Jaipur,market,Gen1,12,0
39,STN-DEL-040,40335,36848,8.65,Delhi NCR,market,Gen1,16,6
49,STN-DEL-050,47060,43007,8.61,Delhi NCR,commercial_hub,Gen1,12,0


In [ ]:
generation_performance = (
    station_performance
    .groupby("charger_generation")
    .agg(
        stations=("station_id", "count"),
        attempts=("attempts", "sum"),
        completed=("completed", "sum")
    )
    .reset_index()
)

generation_performance["failure_rate"] = (
    (generation_performance["attempts"] - generation_performance["completed"])
    / generation_performance["attempts"] * 100
).round(2)

generation_performance

,charger_generation,stations,attempts,completed,failure_rate
0,Gen1,51,1822034,1690298,7.23
1,Gen2,64,1682738,1601293,4.84
2,Gen3,37,372241,354218,4.84


In [ ]:
station_performance["total_slots"] = (
    station_performance["slots_2w"].fillna(0) +
    station_performance["slots_3w"].fillna(0)
)

station_performance["capacity_group"] = pd.cut(
    station_performance["total_slots"],
    bins=[0, 10, 16, 25, 100],
    labels=["Low", "Medium", "High", "Very High"]
)

capacity_performance = (
    station_performance.groupby("capacity_group", observed=True)
    .agg(
        stations=("station_id", "count"),
        attempts=("attempts", "sum"),
        completed=("completed", "sum")
    )
    .reset_index()
)

capacity_performance["failure_rate"] = (
    (capacity_performance["attempts"] - capacity_performance["completed"])
    / capacity_performance["attempts"] * 100
).round(2)

capacity_performance

,capacity_group,stations,attempts,completed,failure_rate
0,Low,27,658481,622582,5.45
1,Medium,90,2295737,2154927,6.13
2,High,35,922795,868300,5.91


In [ ]:
queue_analysis = swaps.groupby("event_type")["queue_wait_sec"].agg(
    ["count", "mean", "median", "max"]
).round(2)

queue_analysis

,count,mean,median,max
event_type,,,,
abandoned_queue,68533,702.56,648.0,2400
cancelled_by_rider,16712,44.57,44.0,89
failed_no_charged_battery,134389,29.50,30.0,59
failed_system_error,11570,44.42,44.0,89
swap_completed,3645809,243.72,209.0,1800


In [ ]:
swaps["hour"] = swaps["event_ts"].dt.hour

hourly = swaps.groupby("hour").agg(
    attempts=("event_id", "count"),
    completed=("event_type", lambda x: (x == "swap_completed").sum())
).reset_index()

hourly["failure_rate"] = (
    (hourly["attempts"] - hourly["completed"])
    / hourly["attempts"] * 100
).round(2)

hourly

,hour,attempts,completed,failure_rate
0,0,23043,21768,5.53
1,1,23482,22190,5.50
2,2,25536,24094,5.65
3,3,28822,27309,5.25
4,4,31193,29603,5.10
5,5,28510,26990,5.33
6,6,62020,58626,5.47
7,7,66889,63248,5.44
8,8,171129,161445,5.66
9,9,232933,219890,5.60


In [ ]:
city_hour = (
    swaps.groupby(["hour", "station_id"])
    .agg(
        attempts=("event_id", "count"),
        completed=("event_type", lambda x: (x == "swap_completed").sum())
    )
    .reset_index()
)

city_hour = city_hour.merge(
    stations[["station_id", "city"]],
    on="station_id",
    how="left"
)

city_hour = (
    city_hour.groupby(["city", "hour"])
    .agg(
        attempts=("attempts", "sum"),
        completed=("completed", "sum")
    )
    .reset_index()
)

city_hour["failure_rate"] = (
    (city_hour["attempts"] - city_hour["completed"])
    / city_hour["attempts"] * 100
).round(2)

city_hour.sort_values("failure_rate", ascending=False).head(15)

,city,hour,attempts,completed,failure_rate
94,Jaipur,22,21904,19776,9.72
93,Jaipur,21,44346,40142,9.48
92,Jaipur,20,56714,51346,9.47
91,Jaipur,19,56926,51627,9.31
44,Delhi NCR,20,89067,81248,8.78
46,Delhi NCR,22,36038,32887,8.74
45,Delhi NCR,21,71009,64828,8.70
43,Delhi NCR,19,89309,81608,8.62
67,Hyderabad,19,82351,75868,7.87
70,Hyderabad,22,32799,30244,7.79


In [ ]:
evening = swaps[
    (swaps["event_ts"].dt.hour >= 19) &
    (swaps["event_ts"].dt.hour <= 22)
]

evening_city_failure = (
    evening.groupby(["event_type"])
    .size()
    .reset_index(name="count")
)

evening_city_failure["percentage"] = (
    evening_city_failure["count"] / len(evening) * 100
).round(2)

evening_city_failure

,event_type,count,percentage
0,abandoned_queue,27819,1.95
1,cancelled_by_rider,6135,0.43
2,failed_no_charged_battery,56079,3.92
3,failed_system_error,4274,0.30
4,swap_completed,1335330,93.40


In [ ]:
evening_city = swaps[
    (swaps["event_ts"].dt.hour >= 19) &
    (swaps["event_ts"].dt.hour <= 22)
].copy()

evening_city = evening_city.merge(
    stations[["station_id", "city"]],
    on="station_id",
    how="left"
)

result = (
    evening_city.groupby("city")
    .agg(
        attempts=("event_id", "count"),
        battery_failures=("event_type",
                         lambda x: (x == "failed_no_charged_battery").sum()),
        queue_abandoned=("event_type",
                         lambda x: (x == "abandoned_queue").sum())
    )
    .reset_index()
)

result["battery_failure_rate"] = (
    result["battery_failures"] / result["attempts"] * 100
).round(2)

result["queue_abandon_rate"] = (
    result["queue_abandoned"] / result["attempts"] * 100
).round(2)

result.sort_values("battery_failure_rate", ascending=False)

,city,attempts,battery_failures,queue_abandoned,battery_failure_rate,queue_abandon_rate
3,Jaipur,179890,10674,5038,5.93,2.80
1,Delhi NCR,285423,15425,7356,5.40,2.58
2,Hyderabad,262930,12400,6116,4.72,2.33
5,Pune,204216,5870,2985,2.87,1.46
0,Bengaluru,303939,7386,3945,2.43,1.30
4,Mumbai,193239,4324,2379,2.24,1.23


In [ ]:
status = station_status.copy()

status["hour_start"] = pd.to_datetime(status["hour_start"])
status["hour"] = status["hour_start"].dt.hour

evening_status = status[
    status["hour"].between(19, 22)
]

stock_analysis = (
    evening_status.groupby("station_id")
    .agg(
        avg_charged_stock=("charged_2w_avg", "mean"),
        min_charged_stock=("charged_2w_min", "mean"),
        avg_outage=("outage_minutes", "mean"),
        avg_charging=("packs_charging", "mean")
    )
    .reset_index()
)

stock_analysis = stock_analysis.merge(
    stations[["station_id", "city"]],
    on="station_id",
    how="left"
)

stock_analysis.groupby("city").agg(
    avg_charged_stock=("avg_charged_stock", "mean"),
    avg_min_stock=("min_charged_stock", "mean"),
    avg_outage=("avg_outage", "mean")
).round(2)

,avg_charged_stock,avg_min_stock,avg_outage
city,,,
Bengaluru,8.32,6.82,0.00
Delhi NCR,8.54,7.03,0.01
Hyderabad,8.78,7.28,0.01
Jaipur,8.77,7.28,0.01
Mumbai,9.55,8.05,0.02
Pune,10.15,8.65,0.01


In [ ]:
station_failure = (
    swaps.groupby("station_id")
    .agg(
        total_attempts=("event_id", "count"),
        failures=("event_type", lambda x: (x != "swap_completed").sum())
    )
    .reset_index()
)

station_failure["failure_rate"] = (
    station_failure["failures"] /
    station_failure["total_attempts"] * 100
)

station_stock = (
    evening_status.groupby("station_id")
    .agg(
        avg_charged_stock=("charged_2w_avg", "mean"),
        avg_min_stock=("charged_2w_min", "mean")
    )
    .reset_index()
)

stock_failure = station_failure.merge(
    station_stock,
    on="station_id",
    how="inner"
)

stock_failure[[
    "avg_min_stock",
    "failure_rate"
]].corr()

,avg_min_stock,failure_rate
avg_min_stock,1.000000,-0.128118
failure_rate,-0.128118,1.000000


In [ ]:
gen_evening = swaps[
    swaps["event_ts"].dt.hour.between(19, 22)
].merge(
    stations[["station_id", "charger_generation"]],
    on="station_id",
    how="left"
)

gen_evening_analysis = (
    gen_evening.groupby("charger_generation")
    .agg(
        attempts=("event_id", "count"),
        failures=("event_type", lambda x: (x != "swap_completed").sum())
    )
    .reset_index()
)

gen_evening_analysis["failure_rate"] = (
    gen_evening_analysis["failures"] /
    gen_evening_analysis["attempts"] * 100
)

gen_evening_analysis.round(2)

,charger_generation,attempts,failures,failure_rate
0,Gen1,677278,57867,8.54
1,Gen2,617301,29874,4.84
2,Gen3,135058,6566,4.86


In [ ]:
gen1_stations = station_failure.merge(
    stations[["station_id", "city", "charger_generation"]],
    on="station_id",
    how="left"
)

gen1_top = (
    gen1_stations[
        gen1_stations["charger_generation"] == "Gen1"
    ]
    .sort_values("failure_rate", ascending=False)
    [["station_id", "city", "total_attempts", "failures", "failure_rate"]]
    .head(15)
)

gen1_top.round(2)

,station_id,city,total_attempts,failures,failure_rate
36,STN-DEL-037,Delhi NCR,30838,2799,9.08
47,STN-DEL-048,Delhi NCR,34646,3111,8.98
92,STN-JAI-137,Jaipur,44371,3881,8.75
97,STN-JAI-142,Jaipur,33319,2914,8.75
96,STN-JAI-141,Jaipur,45594,3981,8.73
44,STN-DEL-045,Delhi NCR,32237,2811,8.72
91,STN-JAI-136,Jaipur,29340,2555,8.71
93,STN-JAI-138,Jaipur,48945,4242,8.67
39,STN-DEL-040,Delhi NCR,40335,3487,8.65
49,STN-DEL-050,Delhi NCR,47060,4053,8.61


In [ ]:
problem_stations = gen1_top["station_id"].tolist()

root_cause = (
    swaps[swaps["station_id"].isin(problem_stations)]
    .groupby(["station_id", "event_type"])
    .size()
    .reset_index(name="count")
)

root_cause["failure_share"] = (
    root_cause.groupby("station_id")["count"]
    .transform(lambda x: x / x.sum() * 100)
)

root_cause = root_cause.sort_values(
    ["station_id", "count"],
    ascending=[True, False]
)

root_cause.round(2)

,station_id,event_type,count,failure_share
4,STN-DEL-036,swap_completed,32468,91.46
2,STN-DEL-036,failed_no_charged_battery,1843,5.19
0,STN-DEL-036,abandoned_queue,929,2.62
1,STN-DEL-036,cancelled_by_rider,153,0.43
3,STN-DEL-036,failed_system_error,106,0.30
...,...,...,...,...
74,STN-JAI-142,swap_completed,30405,91.25
72,STN-JAI-142,failed_no_charged_battery,1740,5.22
70,STN-JAI-142,abandoned_queue,938,2.82
71,STN-JAI-142,cancelled_by_rider,136,0.41


In [ ]:
city_revenue = (
    swaps.groupby("station_id")
    .agg(
        total_attempts=("event_id", "count"),
        completed_swaps=("event_type", lambda x: (x == "swap_completed").sum()),
        revenue=("amount_charged_inr", "sum")
    )
    .reset_index()
)

city_revenue = city_revenue.merge(
    stations[["station_id", "city"]],
    on="station_id",
    how="left"
)

city_revenue_summary = (
    city_revenue.groupby("city")
    .agg(
        total_attempts=("total_attempts", "sum"),
        completed_swaps=("completed_swaps", "sum"),
        revenue=("revenue", "sum")
    )
    .reset_index()
)

city_revenue_summary["revenue_per_attempt"] = (
    city_revenue_summary["revenue"] /
    city_revenue_summary["total_attempts"]
)

city_revenue_summary["revenue_per_completed_swap"] = (
    city_revenue_summary["revenue"] /
    city_revenue_summary["completed_swaps"]
)

city_revenue_summary.round(2)

,city,total_attempts,completed_swaps,revenue,revenue_per_attempt,revenue_per_completed_swap
0,Bengaluru,825327,787697,51954743.25,62.95,65.96
1,Delhi NCR,768913,712385,44565107.85,57.96,62.56
2,Hyderabad,711930,664312,41521622.25,58.32,62.50
3,Jaipur,487634,449366,28229052.85,57.89,62.82
4,Mumbai,530299,506718,32089247.45,60.51,63.33
5,Pune,552910,525331,34766281.90,62.88,66.18


In [ ]:
[c for c in swaps.columns if any(
    x in c.lower()
    for x in ["cost", "margin", "energy", "price", "discount"]
)]

['list_price_inr', 'discount_inr', 'energy_to_recharge_kwh']

In [ ]:
discount_analysis = swaps[
    swaps["event_type"] == "swap_completed"
].copy()

discount_analysis["discount_pct"] = (
    discount_analysis["discount_inr"] /
    discount_analysis["list_price_inr"] * 100
)

discount_analysis[[
    "list_price_inr",
    "discount_inr",
    "amount_charged_inr",
    "discount_pct",
    "energy_to_recharge_kwh"
]].describe().round(2)

,list_price_inr,discount_inr,amount_charged_inr,discount_pct,energy_to_recharge_kwh
count,3645809.0,3645809.00,3645809.00,3645809.00,3645809.00
mean,68.6,5.82,63.94,8.48,2.09
std,13.6,5.62,14.55,8.12,0.81
min,60.0,0.00,36.80,0.00,0.81
25%,65.0,0.00,55.25,0.00,1.72
50%,65.0,6.50,59.80,9.00,1.87
75%,65.0,7.80,65.00,12.00,2.03
max,110.0,30.80,135.00,28.00,5.52


In [ ]:
city_discount = (
    discount_analysis
    .merge(
        stations[["station_id", "city"]],
        on="station_id",
        how="left"
    )
    .groupby("city")
    .agg(
        avg_list_price=("list_price_inr", "mean"),
        avg_discount=("discount_inr", "mean"),
        avg_amount_charged=("amount_charged_inr", "mean"),
        avg_discount_pct=("discount_pct", "mean"),
        avg_energy_kwh=("energy_to_recharge_kwh", "mean")
    )
    .reset_index()
)

city_discount.round(2)

,city,avg_list_price,avg_discount,avg_amount_charged,avg_discount_pct,avg_energy_kwh
0,Bengaluru,68.58,5.89,65.96,8.57,2.10
1,Delhi NCR,68.37,5.81,62.56,8.51,2.07
2,Hyderabad,68.29,5.79,62.50,8.46,2.08
3,Jaipur,68.45,5.64,62.82,8.20,2.10
4,Mumbai,69.31,5.99,63.33,8.66,2.12
5,Pune,68.77,5.79,66.18,8.44,2.09


In [ ]:
riders.columns.tolist()

['rider_id',
 'partner_id',
 'vehicle_class',
 'vehicle_model',
 'home_zone',
 'signup_date',
 'signup_channel',
 'plan_type',
 'declared_shift',
 'age_band',
 'kyc_verified',
 'home_city']

In [ ]:
riders.head()

,rider_id,partner_id,vehicle_class,vehicle_model,home_zone,signup_date,signup_channel,plan_type,declared_shift,age_band,kyc_verified,home_city
0,RD-1000000,FP-04,3W,Kranti Cargo3,DEL-Noida,2023-03-08,field_agent,partner_billed,night,45+,True,Delhi NCR
1,RD-1000001,FP-05,2W,Volt Cub,MUM-Thane,2023-03-08,partner_onboarding,partner_billed,mixed,35-44,True,Mumbai
2,RD-1000002,FP-06,2W,Torq X1,HYD-West,2023-03-08,partner_onboarding,partner_billed,evening,25-34,False,Hyderabad
3,RD-1000003,FP-01,2W,Zipp E2,MUM-Navi Mumbai,2023-03-08,partner_onboarding,partner_billed,evening,25-34,True,Mumbai
4,RD-1000004,FP-02,2W,Zipp E2,HYD-Hitech City,2023-03-08,partner_onboarding,partner_billed,night,NaN,True,Hyderabad


In [ ]:
rider_activity = (
    swaps.groupby("rider_id")
    .agg(
        total_attempts=("event_id", "count"),
        completed_swaps=("event_type", lambda x: (x == "swap_completed").sum()),
        last_activity=("event_ts", "max"),
        first_activity=("event_ts", "min")
    )
    .reset_index()
)

rider_activity.head()

,rider_id,total_attempts,completed_swaps,last_activity,first_activity
0,RD-1000000,235,209,2024-08-10 17:38:38,2024-01-03 10:51:42
1,RD-1000001,165,152,2024-10-06 20:17:53,2024-01-02 19:26:59
2,RD-1000002,787,740,2025-06-30 08:45:06,2024-01-03 09:24:12
3,RD-1000003,121,117,2024-05-05 19:39:08,2024-01-01 11:10:56
4,RD-1000004,440,422,2025-03-08 18:40:29,2024-01-01 17:13:06


In [ ]:
analysis_end = swaps["event_ts"].max()

rider_activity["days_since_last_activity"] = (
    analysis_end - rider_activity["last_activity"]
).dt.days

rider_activity["churn_flag"] = (
    rider_activity["days_since_last_activity"] >= 30
)

rider_activity["churn_flag"].value_counts()

,count
churn_flag,
False,12030
True,7920


In [ ]:
rider_churn = rider_activity.merge(
    riders,
    on="rider_id",
    how="left"
)

churn_by_vehicle = (
    rider_churn.groupby("vehicle_class")
    .agg(
        riders=("rider_id", "count"),
        churned=("churn_flag", "sum")
    )
    .reset_index()
)

churn_by_vehicle["churn_rate"] = (
    churn_by_vehicle["churned"] /
    churn_by_vehicle["riders"] * 100
)

churn_by_vehicle.round(2)

,vehicle_class,riders,churned,churn_rate
0,2W,17070,6725,39.40
1,3W,2880,1195,41.49


In [ ]:
churn_by_plan = (
    rider_churn.groupby("plan_type")
    .agg(
        riders=("rider_id", "count"),
        churned=("churn_flag", "sum")
    )
    .reset_index()
)

churn_by_plan["churn_rate"] = (
    churn_by_plan["churned"] /
    churn_by_plan["riders"] * 100
)

churn_by_plan.round(2)

,plan_type,riders,churned,churn_rate
0,partner_billed,12348,4942,40.02
1,pay_as_you_go,5285,2072,39.21
2,prepaid_pack,2317,906,39.10


In [ ]:
churn_by_city = (
    rider_churn.groupby("home_city")
    .agg(
        riders=("rider_id", "count"),
        churned=("churn_flag", "sum")
    )
    .reset_index()
)

churn_by_city["churn_rate"] = (
    churn_by_city["churned"] /
    churn_by_city["riders"] * 100
)

churn_by_city.sort_values(
    "churn_rate",
    ascending=False
).round(2)

,home_city,riders,churned,churn_rate
6,Gurgaon,36,20,55.56
0,BLR,53,28,52.83
18,hyderabad,27,13,48.15
15,PUN,42,19,45.24
11,Jaipur,2444,1046,42.80
8,Hyd,31,13,41.94
5,Delhi NCR,3777,1568,41.51
14,New Delhi,42,17,40.48
3,Bombay,53,21,39.62
9,Hyderabad,3389,1340,39.54


In [ ]:
city_map = {
    "BLR": "Bengaluru",
    "Bangalore": "Bengaluru",
    "bengaluru": "Bengaluru",

    "Hyd": "Hyderabad",
    "hyd": "Hyderabad",
    "HYD": "Hyderabad",
    "hyderabad": "Hyderabad",

    "PUN": "Pune",
    "pune": "Pune",

    "Bombay": "Mumbai",
    "MUM": "Mumbai",

    "JAI": "Jaipur",
    "jaipur": "Jaipur",

    "Delhi": "Delhi NCR",
    "New Delhi": "Delhi NCR",
    "Gurgaon": "Delhi NCR"
}

rider_churn["city_clean"] = (
    rider_churn["home_city"]
    .map(city_map)
    .fillna(rider_churn["home_city"])
)

churn_by_city_clean = (
    rider_churn.groupby("city_clean")
    .agg(
        riders=("rider_id", "count"),
        churned=("churn_flag", "sum")
    )
    .reset_index()
)

churn_by_city_clean["churn_rate"] = (
    churn_by_city_clean["churned"] /
    churn_by_city_clean["riders"] * 100
)

churn_by_city_clean.sort_values(
    "riders",
    ascending=False
).round(2)

,city_clean,riders,churned,churn_rate
0,Bengaluru,4247,1634,38.47
1,Delhi NCR,3880,1610,41.49
2,Hyderabad,3478,1378,39.62
5,Pune,2917,1143,39.18
4,Mumbai,2864,1072,37.43
3,Jaipur,2514,1067,42.44
6,bengaluru,50,16,32.00


In [ ]:
rider_experience = (
    swaps.groupby("rider_id")
    .agg(
        total_attempts=("event_id", "count"),
        failures=("event_type", lambda x: (x != "swap_completed").sum()),
        avg_queue_wait=("queue_wait_sec", "mean")
    )
    .reset_index()
)

rider_experience["failure_rate"] = (
    rider_experience["failures"] /
    rider_experience["total_attempts"] * 100
)

churn_experience = rider_churn[
    ["rider_id", "churn_flag"]
].merge(
    rider_experience,
    on="rider_id",
    how="left"
)

churn_experience.groupby("churn_flag").agg(
    riders=("rider_id", "count"),
    avg_failure_rate=("failure_rate", "mean"),
    avg_queue_wait=("avg_queue_wait", "mean"),
    avg_attempts=("total_attempts", "mean")
).round(2)

,riders,avg_failure_rate,avg_queue_wait,avg_attempts
churn_flag,,,,
False,12030,6.90,242.88,225.74
True,7920,6.43,243.46,146.64


In [ ]:
swaps["event_ts"] = pd.to_datetime(swaps["event_ts"])

swaps["month"] = swaps["event_ts"].dt.to_period("M").astype(str)

monthly_rider_activity = (
    swaps.groupby("month")
    .agg(
        active_riders=("rider_id", "nunique"),
        total_attempts=("event_id", "count"),
        completed_swaps=("event_type", lambda x: (x == "swap_completed").sum())
    )
    .reset_index()
)

monthly_rider_activity["attempts_per_rider"] = (
    monthly_rider_activity["total_attempts"] /
    monthly_rider_activity["active_riders"]
)

monthly_rider_activity["completed_per_rider"] = (
    monthly_rider_activity["completed_swaps"] /
    monthly_rider_activity["active_riders"]
)

monthly_rider_activity.round(2)

,month,active_riders,total_attempts,completed_swaps,attempts_per_rider,completed_per_rider
0,2024-01,4186,110348,105490,26.36,25.20
1,2024-02,4450,111531,106582,25.06,23.95
2,2024-03,4741,123900,118444,26.13,24.98
3,2024-04,5081,131818,122836,25.94,24.18
4,2024-05,5489,146278,128328,26.65,23.38
5,2024-06,5846,148888,133743,25.47,22.88
6,2024-07,6178,158688,151824,25.69,24.57
7,2024-08,6717,172295,164931,25.65,24.55
8,2024-09,7242,195885,187302,27.05,25.86
9,2024-10,7941,229901,219838,28.95,27.68


In [ ]:
churn_profile = rider_churn.groupby(
    ["signup_channel"]
).agg(
    riders=("rider_id", "count"),
    churned=("churn_flag", "sum")
).reset_index()

churn_profile["churn_rate"] = (
    churn_profile["churned"] /
    churn_profile["riders"] * 100
)

churn_profile.sort_values(
    "churn_rate",
    ascending=False
).round(2)

,signup_channel,riders,churned,churn_rate
2,partner_onboarding,9359,3757,40.14
3,referral,2259,895,39.62
1,field_agent,4133,1625,39.32
0,app_store,4199,1643,39.13


In [ ]:
ticket_counts = (
    tickets.groupby("rider_id")
    .agg(
        total_tickets=("ticket_id", "count"),
        avg_csat=("csat_score", "mean")
    )
    .reset_index()
)

churn_ticket = rider_churn[
    ["rider_id", "churn_flag"]
].merge(
    ticket_counts,
    on="rider_id",
    how="left"
)

churn_ticket["total_tickets"] = churn_ticket["total_tickets"].fillna(0)

churn_ticket.groupby("churn_flag").agg(
    riders=("rider_id", "count"),
    avg_tickets=("total_tickets", "mean"),
    avg_csat=("avg_csat", "mean")
).round(2)

,riders,avg_tickets,avg_csat
churn_flag,,,
False,12030,2.69,3.56
True,7920,1.47,3.53


## Key Findings

### 1. Overall Network Performance
- 3.88M total swap attempts
- 3.65M completed swaps
- Overall failure rate: 5.96%

### 2. Major Failure Reasons
- No charged battery: 58.13%
- Abandoned queue: 29.64%
- Together: 87.77% of failures

### 3. City-wise Performance
- Jaipur: 7.85% failure rate
- Delhi NCR: 7.35%
- Mumbai: 4.45%

### 4. Charger Generation
- Gen1: 7.23%
- Gen2: 4.84%
- Gen3: 4.84%

### 5. Peak Failure Hours
- 19:00–22:00 shows higher failure rates
- 21:00: 6.65%

### 6. Problem Stations
- STN-DEL-037: 9.08% failure rate
- Top problem stations are concentrated in Delhi NCR and Jaipur

## Actionable Recommendations

### 1. Improve Battery Availability
Focus on stations with high `failed_no_charged_battery` rates.
Maintain adequate charged-battery inventory during peak hours.

### 2. Reduce Queue Waiting Time
Investigate stations with high queue abandonment.
Optimize battery allocation and service capacity during 19:00–22:00.

### 3. Prioritize High-Failure Cities
Conduct detailed operational analysis for Jaipur and Delhi NCR,
where failure rates are higher than other cities.

### 4. Investigate Gen1 Chargers
Review maintenance, hardware condition and operational performance
of Gen1 chargers because they show a higher associated failure rate.

### 5. Target Problem Stations
Prioritize high-failure stations such as STN-DEL-037 and other
top-performing problem stations for root-cause investigation.

### 6. Monitor Peak Hours
Create an operational monitoring system for evening hours,
especially around 19:00–22:00.


## Conclusion

The analysis of the VoltRelay battery-swapping network shows that
service failures are mainly associated with battery availability
and queue abandonment.

Jaipur and Delhi NCR show higher failure rates, while Gen1 chargers
show a higher associated failure rate than Gen2 and Gen3.

The analysis also identifies evening hours and specific high-failure
stations as important areas for operational attention.

These findings can help VoltRelay prioritize battery inventory,
queue management, charger maintenance and station-level improvements.

# VoltRelay Energy – Battery Swapping Network Analytics

## 1. Executive Summary

This project analyzes the performance of VoltRelay Energy's
battery-swapping network using operational, station, battery,
rider and support data.

The analysis covers approximately 3.9 million swap attempts
across six Indian cities from January 2024 to June 2025.

The objective is to identify major service-failure patterns,
station and city-level performance differences, charger-generation
performance and operational areas requiring improvement.

Key findings indicate that battery availability and queue
abandonment account for the majority of recorded failures.
Higher failure rates are also observed in Jaipur and Delhi NCR,
while Gen1 chargers show a higher associated failure rate.

## 2. Business Problem

VoltRelay Energy operates a battery-swapping network across
Bengaluru, Delhi NCR, Hyderabad, Pune, Mumbai and Jaipur.

The business needs to understand how the network is performing,
where service failures are concentrated, which operational factors
are associated with failures, and which areas may require
improvement.

The analysis focuses on:
- Network performance over time
- Service failures and customer experience
- Station and city-level performance
- Charger-generation performance
- Rider retention and churn patterns
- Pricing and operational performance

## 3. Dataset and Analytical Approach

### Dataset

The analysis uses approximately 3.9 million swap attempts
recorded between January 2024 and June 2025 across six cities.

The dataset contains eight related files covering:
- Swap events
- Station hourly status
- Riders
- Batteries
- Support tickets
- Stations
- City daily context
- Fleet partners

### Analytical Approach

The analysis was performed using Python, Pandas and Power BI.

The workflow included:
1. Data loading and understanding
2. Data quality checking
3. KPI calculation
4. Time-based failure analysis
5. City and station-level analysis
6. Charger-generation comparison
7. Rider churn analysis
8. Dashboard development
9. Key findings and actionable recommendations

## 4. Data Cleaning and Quality Checks

Before analysis, the datasets were reviewed for data quality issues
that could affect the results.

The following checks were considered:

- Validation of swap event types and outcomes
- Missing values in important operational and telemetry fields
- Duplicate and near-duplicate swap records
- Invalid or extreme distance values
- Inconsistent rider home-city labels
- Timestamp consistency across records
- Missing customer satisfaction (CSAT) values
- Identification of test stations and unusual operational records

These checks were important to avoid interpreting data-quality issues
as actual business performance problems.

The analysis focuses on completed and failed swap events and uses
aggregated metrics for city, station, time and charger-generation
comparisons.

## 5. KPI and Network Performance

The overall network performance was evaluated using key operational
metrics.

### Key KPIs

- Total swap attempts: 3,877,013
- Completed swaps: 3,645,809
- Overall failure rate: 5.96%
- Completed swap rate: 94.04%

The network handled a large volume of swap activity during the
analysis period. Although the overall completion rate was high,
service failures were concentrated in specific cities, stations,
time periods and charger generations.

These KPIs provide the baseline for detailed failure analysis in the
following sections.


## 6. Service Failure Analysis

Service failures were analyzed by failure type to identify the major
sources of customer experience problems.

### Failure Breakdown

- Failed due to no charged battery: 58.13% of failures
- Abandoned queue: 29.64%
- Cancelled by rider: 7.23%
- System error: 5.00%

The two major failure categories — battery availability and queue
abandonment — together account for approximately 87.77% of all
failures.

This indicates that operational availability and waiting-time
management are important areas for improving swap service reliability.

## 7. City and Station Performance

Performance was compared across the six operating cities to identify
geographical variation in service reliability.

### City-wise Failure Rate

- Jaipur: 7.85%
- Delhi NCR: 7.35%
- Hyderabad: 6.69%
- Pune: 4.99%
- Bengaluru: 4.56%
- Mumbai: 4.45%

Jaipur and Delhi NCR recorded higher failure rates compared with the
other cities.

### Station-level Performance

Station-level analysis was used to identify locations with consistently
higher failure rates.

Among stations with at least 10,000 attempts, STN-DEL-037 recorded
the highest failure rate at 9.08%.

The high-failure stations were concentrated mainly in Delhi NCR and
Jaipur, indicating that station-level investigation can help
prioritize operational improvements.

## 8. Charger Generation and Peak-Hour Analysis

### Charger Generation

Failure rates were compared across charger generations.

- Gen1: 7.23%
- Gen2: 4.84%
- Gen3: 4.84%

Gen1 chargers show a higher observed failure rate than Gen2 and Gen3.
This is an association in the dataset and should not be interpreted as
proof that charger generation directly causes failures.

### Peak-Hour Analysis

Failure rates were also analyzed by hour of the day.

The highest observed failure rate was recorded at:

- 21:00: 6.65%
- 20:00: 6.60%
- 22:00: 6.58%
- 19:00: 6.56%

The 19:00–22:00 period shows relatively higher failure activity,
making evening operations an important period for further investigation.

## 9. Rider Retention and Churn Analysis

Rider activity was analyzed to understand retention patterns.

Since the dataset does not contain an official churn label, an
analytical churn definition was used:

A rider was considered churned if their last recorded activity occurred
at least 30 days before the end of the dataset period.

### Churn Overview

- Riders with swap activity: 19,950
- Retained riders: 12,030
- Churned riders: 7,920
- Analytical churn rate: 39.60%

### Rider Behaviour

Churned riders recorded fewer average swap attempts than retained
riders:

- Retained: 225.74 average attempts
- Churned: 146.64 average attempts

However, average failure rate and queue waiting time were very similar
between the two groups.

Therefore, the analysis indicates lower rider activity among churned
riders, but does not establish that service failures directly caused
churn.

## 10. Revenue and Pricing Analysis

Revenue and pricing patterns were analyzed across the six cities to
understand whether pricing differences were associated with network
performance.

### City-wise Revenue

The network generated approximately ₹233.13 million in revenue from
swap activity during the analysis period.

Revenue per attempt varied across cities, with Bengaluru and Pune
showing higher revenue per attempt than several other cities.

### Pricing Pattern

The average discount on completed swaps was broadly similar across
cities, with discount percentages generally around 8–9%.

This indicates that large pricing differences were not evident across
cities in the available data.

The analysis therefore focuses primarily on operational factors such
as battery availability, queue abandonment, station performance and
charger generation rather than treating pricing as the main explanation
for service failures.

## 11. Key Findings

The analysis produced the following major findings:

1. The network processed 3.88 million swap attempts with a 5.96%
   overall failure rate.

2. Battery availability and queue abandonment were the major failure
   categories, together accounting for 87.77% of all failures.

3. Jaipur and Delhi NCR recorded the highest city-level failure rates.

4. Gen1 chargers recorded a higher observed failure rate (7.23%)
   compared with Gen2 and Gen3 (4.84%).

5. Evening hours, particularly 19:00–22:00, showed relatively higher
   failure activity, with 21:00 recording the highest hourly failure
   rate of 6.65%.

6. High-failure stations were concentrated mainly in Delhi NCR and
   Jaipur, making station-level analysis important for operational
   improvement.

7. Churned riders showed lower average activity than retained riders,
   but the analysis does not establish a direct causal relationship
   between service failures and churn.

## 12. Actionable Recommendations

Based on the analysis, the following operational actions can be
considered:

### 1. Improve Battery Availability
Monitor charged-battery inventory at stations, especially during
high-demand periods, to reduce failures caused by unavailable
charged batteries.

### 2. Reduce Queue Waiting Time
Analyze stations with high queue abandonment and improve operational
capacity during peak evening hours.

### 3. Prioritize High-Failure Cities
Focus detailed operational investigation on Jaipur and Delhi NCR,
where higher failure rates were observed.

### 4. Investigate Gen1 Chargers
Review maintenance history, hardware condition and operational
performance of Gen1 chargers to understand their higher observed
failure rate.

### 5. Target High-Failure Stations
Create a station-level action list for locations with consistently
high failure rates and significant swap volumes.

### 6. Monitor Peak Hours
Use hourly performance monitoring to prepare inventory and operational
resources for the 19:00–22:00 period.

### 7. Track Rider Engagement
Monitor rider activity over time to identify declining usage patterns
and potential retention risks.

## 13. Conclusion

The VoltRelay Energy analysis provides an overall view of battery
swapping network performance across six cities.

The analysis shows that service failures are mainly associated with
charged-battery availability and queue abandonment. Higher failure
rates were observed in Jaipur and Delhi NCR, while Gen1 chargers also
showed higher observed failure rates.

Station-level and peak-hour analysis further helps identify where
operational attention can be prioritized.

Overall, the analysis demonstrates how operational data can be used to
identify performance gaps, compare network segments and support
data-driven operational decision-making.

The findings represent associations observed in the available dataset
and should not be interpreted as proof of causation.

## 14. Limitations and Future Scope

### Limitations

- The dataset is synthetic and represents simulated network activity.
- No official churn label is available; churn was defined analytically
  using rider inactivity.
- The analysis identifies associations and does not establish
  causation.
- Some telemetry and customer-support fields contain missing values.
- Inconsistent rider city labels and timestamp-related data-quality
  issues require careful interpretation.
- Direct contribution margin could not be calculated because complete
  cost information was not available in the analyzed data.

### Future Scope

- Develop predictive models for station-level failure risk.
- Build an early-warning system for battery availability shortages.
- Forecast demand by station and time period.
- Develop a rider retention prediction model.
- Integrate real-time station and battery monitoring.
- Extend the Power BI dashboard with automated data refresh and
  operational alerts.